# NPTEL Course: Grid Connected Power Converters - Operating Principles

## Single-Phase Converter with Switching Pole and Ideal Balanced DC Link

**Author:** V. Seshadri Sravan Kumar, IIT Hyderabad

### Objective and Modelling Scope

The goal of this simulation is to study the switched model of an ideal single-phase converter interfaced with the grid. In this notebook, the DC link is assumed perfectly balanced for all time:

$$v_{C1}(t)=v_{C2}(t)=\frac{V_{dc}}{2}.$$

Hence, DC-link dynamics are not modeled, and the system reduces to a single state variable, the filter current $i_f(t)$.

The **modelling assumptions** are:

1. Balanced ideal DC link.
2. Ideal switching devices.
3. Grid represented as a stiff sinusoidal voltage source.
4. Saturation, and higher-order parasitic effects are neglected.

### Governing Equation

The pole voltage is $$v_{ao}(t)=\frac{V_{dc}}{2}q(t),$$

where $q(t)\in\{+1,-1\}$ is generated by comparing the modulating signal $m(t)=M\sin(\omega_1 t+\phi)$ with the triangular carrier. The AC-side current dynamics are

$$\frac{di_f}{dt}=\frac{1}{L}\left[\frac{V_{dc}}{2}q(t)-Ri_f(t)-v_g(t)\right].$$

### What Is Studied in This Notebook

1. Switching function generation from PWM comparison.
2. Time-domain waveforms of $i_f(t)$ and $v_{ao}(t)$.
3. Rail current waveforms $i_t(t)$ and $i_b(t)$.
4. Harmonic spectrum and THD of pole voltage and injected current using DFT.

In [ ]:
# Importing required packages

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Plotting style settings (applied globally to all subsequent plots)
plt.rcParams['figure.figsize']    = (10, 5)
plt.rcParams['axes.grid']         = True
plt.rcParams['grid.alpha']        = 0.3
plt.rcParams['font.size']         = 10
#
plt.rcParams['axes.spines.top']   = True
plt.rcParams['axes.spines.right'] = True
plt.rcParams['xtick.top']         = True
plt.rcParams['ytick.right']       = True
plt.rcParams['xtick.direction']   = 'in'
plt.rcParams['ytick.direction']   = 'in'

### System Parameters

The parameter values below represent a single-phase grid-connected converter rated at 3 kW, connected to a 220 V, 50 Hz grid.

In [ ]:
Vg_rms   = 220.0          # grid RMS voltage (V)
f1       = 50.0           # grid fundamental frequency (Hz)

Vdc      = 700.0          # DC link voltage (V), fixed and perfectly balanced

M        = 0.89           # modulation index
phi      = -3*np.pi/4     # phase of modulating signal m(t), in radians

fsw      = 4000.0         # switching frequency (Hz)
m_f      = fsw/f1         # carrier ratio

L        = 45e-3          # filter inductance (H)
R        = 0.15           # filter resistance (ohm)

### Time Discretization

The sampling interval is chosen from the required number of points per switching period. A sufficient number of samples per switching cycle helps capture ripple clearly and accurately.

In [ ]:
n_cycles      = 4                             # number of fundamental cycles to simulate
n_pts_per_Tsw = 40                            # number of time instants per switching cycle

t_end         = n_cycles/f1                   # total simulation time (s)
t_step        = (1.0/fsw)/n_pts_per_Tsw       # time step (s)

t             = np.arange(0, t_end, t_step)   # time vector

### Switching Function and Grid Voltage

The modulation signal, triangular carrier, and sawtooth carrier are defined using `m_signal(t)`, `carrier_triangle(t)`, and `carrier_sawtooth(t)`, respectively.

In `q_signal(t)`, choose the PWM method by commenting/uncommenting the carrier line (sine-triangle PWM or sine-sawtooth PWM).

The grid voltage is defined by `vg_signal(t)`, with grid phase set to 0.

In [ ]:
def carrier_triangle(t):
    """Symmetric triangular carrier, amplitude [-1, +1], period 1/fsw."""
    Tsw = 1.0/fsw
    x = (t % Tsw)/Tsw
    return np.where(x < 0.5, 4*x - 1.0, 3.0 - 4*x)

def carrier_sawtooth(t):
    """Sawtooth carrier, amplitude [-1, +1], period 1/fsw."""
    Tsw = 1.0/fsw
    x = (t % Tsw)/Tsw
    return 2*x - 1.0

def m_signal(t):
    """Modulating reference."""
    return M*np.sin(2*np.pi*f1*t + phi)

def q_signal(t):
    """Switching function q(t) in {+1, -1}."""
    # Choose carrier by comment/uncomment:
    # carrier = carrier_triangle(t)
    carrier = carrier_sawtooth(t)
    return np.where(m_signal(t) >= carrier, 1.0, -1.0)

def vg_signal(t):
    """Grid voltage."""
    return np.sqrt(2)*Vg_rms*np.cos(2*np.pi*f1*t)

### Visualization of the Switching Function

The code below plots the modulation signal, carrier signal, and switching function over one fundamental cycle and over four switching cycles.

In [ ]:
T1  = 1.0/f1
Tsw = 1.0/fsw

# Last 1 fundamental cycle
t_fund = np.linspace(t_end - T1, t_end, 12000)
m_fund = m_signal(t_fund)
c_fund = carrier_sawtooth(t_fund)
q_fund = q_signal(t_fund)

# Last 4 switching cycles
t_zoom = np.linspace(t_end - 4*Tsw, t_end, 3000)
m_zoom = m_signal(t_zoom)
c_zoom = carrier_sawtooth(t_zoom)
q_zoom = q_signal(t_zoom)

fig, axs = plt.subplots(2, 2, figsize=(11, 5), sharex='col', sharey='row')

# Top row: modulation and carrier
axs[0, 0].plot(t_fund*1000, m_fund, color='red', lw=1.0, label='m(t)')
axs[0, 0].plot(t_fund*1000, c_fund, color='blue', lw=1.0, label='c(t)')
axs[0, 0].set_title('Last fundamental cycle')
axs[0, 0].set_ylabel('Amplitude')
axs[0, 0].set_ylim([-1.2, 1.2])
axs[0, 0].legend(loc='upper right')

axs[0, 1].plot(t_zoom*1000, m_zoom, color='red', lw=1.0)
axs[0, 1].plot(t_zoom*1000, c_zoom, color='blue', lw=1.0)
axs[0, 1].set_title('Last 4 switching cycles')
axs[0, 1].set_ylim([-1.2, 1.2])

# Bottom row: switching function
axs[1, 0].step(t_fund*1000, q_fund, where='post', color='black', lw=1.0, label='q(t)')
axs[1, 0].set_ylabel('q(t)')
axs[1, 0].set_xlabel('time (ms)')
axs[1, 0].set_yticks([-1, 1])
axs[1, 0].set_ylim([-1.2, 1.2])
axs[1, 0].legend(loc='upper right')

axs[1, 1].step(t_zoom*1000, q_zoom, where='post', color='black', lw=1.0)
axs[1, 1].set_xlabel('time (ms)')
axs[1, 1].set_yticks([-1, 1])
axs[1, 1].set_ylim([-1.2, 1.2])

plt.tight_layout()
plt.show()

### Initial Condition Selection

Initial condition selection is important. If `i_f(0)` is set to zero, the simulation may need a longer runtime to reach steady state because `R` is small.

To reduce startup transients, `i_f(0)` is initialized from a fundamental-frequency steady-state phasor estimate.

In [ ]:
w1          = 2*np.pi*f1

V_ao_phasor = (Vdc*M/2) * np.exp(1j*(phi - np.pi/2))
V_g_phasor  = np.sqrt(2)*Vg_rms * np.exp(1j*0)

I_f_phasor  = (V_ao_phasor - V_g_phasor) / (R + 1j*w1*L)

x0          = np.array([np.real(I_f_phasor)])

print(f"The initial current through the inductor is i_f(0) = {x0[0]:.3f} A")

### ODE Formulation and Numerical Integration

The state equation contains a single state, $i_f(t)$. Numerical integration is carried out using `solve_ivp`, with `max_step` limited to the chosen simulation step so that switching transitions are properly captured.

In [ ]:
def rhs(t, x):
    i_f  = x[0]
    q    = q_signal(t)
    vg   = vg_signal(t)
    di_f = (Vdc/2*q - R*i_f - vg)/L
    return np.array([di_f])

max_step = t_step
sol      = solve_ivp(rhs, (t[0], t[-1]), x0, max_step=max_step, dense_output=True)

i_f = sol.sol(t)[0]

### Waveforms: Injected Current and Pole Voltage

The two primary signals, $i_f(t)$ and $v_{ao}(t)$, are examined at three time scales: full simulation interval, final two fundamental cycles, and final four switching cycles.

In [ ]:
q_t  = q_signal(t)
v_ao = Vdc/2 * q_t

# Time windows
mask_2fund = (t >= (t_end - 2/f1))
mask_4sw   = (t >= (t_end - 4/fsw))

fig, axs = plt.subplots(2, 3, figsize=(16, 7))

# -------------------------------
# Top row: injected current i_f(t)
# -------------------------------
axs[0, 0].plot(t*1000, i_f, color='blue', lw=1.0)
axs[0, 0].set_title('Complete simulation')
axs[0, 0].set_ylabel('$i_f$ (A)')

axs[0, 1].plot(t[mask_2fund]*1000, i_f[mask_2fund], color='blue', lw=1.0)
axs[0, 1].set_title('Last 2 fundamental cycles')
axs[0, 1].set_ylabel('$i_f$ (A)')

axs[0, 2].plot(t[mask_4sw]*1000, i_f[mask_4sw], color='blue', lw=1.0)
axs[0, 2].set_title('Last 4 switching cycles')
axs[0, 2].set_ylabel('$i_f$ (A)')

# -----------------------------
# Bottom row: pole voltage v_ao(t)
# -----------------------------
axs[1, 0].plot(t*1000, v_ao, color='gray', lw=1.0)
axs[1, 0].set_ylabel('$v_{ao}$ (V)')
axs[1, 0].set_xlabel('time (ms)')

axs[1, 1].plot(t[mask_2fund]*1000, v_ao[mask_2fund], color='gray', lw=1.0)
axs[1, 1].set_ylabel('$v_{ao}$ (V)')
axs[1, 1].set_xlabel('time (ms)')

axs[1, 2].plot(t[mask_4sw]*1000, v_ao[mask_4sw], color='gray', lw=1.0)
axs[1, 2].set_ylabel('$v_{ao}$ (V)')
axs[1, 2].set_xlabel('time (ms)')

plt.tight_layout()
plt.show()

### Rail Current Waveforms

The rail currents are computed as

$$i_t(t)=\frac{1+q(t)}{2}i_f(t), \qquad i_b(t)=\frac{1-q(t)}{2}i_f(t),$$

and plotted at the same three time scales used previously.

In [ ]:
i_t = (1 + q_t)/2 * i_f
i_b = (1 - q_t)/2 * i_f

# Reuse time-window masks defined earlier: mask_2fund, mask_4sw
fig, axs = plt.subplots(2, 3, figsize=(16, 7))

# -------------------------
# Top row: top rail current i_t(t)
# -------------------------
axs[0, 0].plot(t*1000, i_t, color='red', lw=1.0)
axs[0, 0].set_title('Complete simulation')
axs[0, 0].set_ylabel('$i_t$ (A)')

axs[0, 1].plot(t[mask_2fund]*1000, i_t[mask_2fund], color='red', lw=1.0)
axs[0, 1].set_title('Last 2 fundamental cycles')
axs[0, 1].set_ylabel('$i_t$ (A)')

axs[0, 2].plot(t[mask_4sw]*1000, i_t[mask_4sw], color='red', lw=1.0)
axs[0, 2].set_title('Last 4 switching cycles')
axs[0, 2].set_ylabel('$i_t$ (A)')

# ----------------------------
# Bottom row: bottom rail current i_b(t)
# ----------------------------
axs[1, 0].plot(t*1000, i_b, color='green', lw=1.0)
axs[1, 0].set_ylabel('$i_b$ (A)')
axs[1, 0].set_xlabel('time (ms)')

axs[1, 1].plot(t[mask_2fund]*1000, i_b[mask_2fund], color='green', lw=1.0)
axs[1, 1].set_ylabel('$i_b$ (A)')
axs[1, 1].set_xlabel('time (ms)')

axs[1, 2].plot(t[mask_4sw]*1000, i_b[mask_4sw], color='green', lw=1.0)
axs[1, 2].set_ylabel('$i_b$ (A)')
axs[1, 2].set_xlabel('time (ms)')

plt.tight_layout()
plt.show()

### Harmonic Analysis by Discrete Fourier Transform

For harmonic analysis, one steady-state fundamental period, $T_1=1/f_1$, is extracted from each waveform and sampled at $N$ points. The Fourier coefficients are computed using

$$C_k=\frac{1}{N}\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N}.$$

Because the analysis window equals one fundamental period, the DFT bin index $k$ is directly the harmonic order $h$.

For real-valued signals, coefficients are converted to a single-sided peak-amplitude spectrum by taking $\hat C_h=2C_h$ for $h>0$, with standard treatment of the DC and Nyquist bins. Total harmonic distortion is then evaluated as

$$THD=\frac{\sqrt{\sum_{h\ge 2}|\hat C_h|^2}}{|\hat C_1|}.$$

The following code implements this procedure and plots magnitude and phase spectra versus harmonic order.

In [ ]:
def dft(signal):
    """Direct DFT via the DFT matrix."""
    N = len(signal)
    k = np.arange(N)
    n = np.arange(N)
    W = (1/N) * np.exp(-2j * np.pi * np.outer(k, n) / N)
    return W @ signal

def dft_peak(signal):
    """Single-sided peak-amplitude spectrum for a real-valued signal."""
    N = len(signal)
    Ck = dft(signal)
    n_bins = N // 2 + 1
    Ck_p = (2 * Ck[:n_bins]).copy()
    Ck_p[0] /= 2
    if N % 2 == 0:
        Ck_p[-1] /= 2
    return Ck_p

def compute_thd(Ck_p):
    """Total harmonic distortion from the peak-amplitude spectrum."""
    fundamental_magnitude = np.abs(Ck_p[1])
    harmonic_magnitudes = np.abs(Ck_p[2:])
    thd = np.sqrt(np.sum(harmonic_magnitudes**2)) / fundamental_magnitude if fundamental_magnitude > 0 else 0
    print("The total harmonic distortion (THD) is {:.2f}%".format(thd * 100))
    return thd

def plot_spectrum(Ck_p, h_max=200, title=None, unit=''):
    """Plot magnitude and phase spectra side by side, vs. harmonic order h."""
    h = np.arange(len(Ck_p))
    mask = (h <= h_max)

    mag   = np.abs(Ck_p[mask])
    phase = np.degrees(np.angle(Ck_p[mask]))
    phase[mag < 1e-6*np.max(mag)] = 0.0

    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))

    axs[0].stem(h[mask], mag, linefmt='tab:blue', markerfmt='o', basefmt=' ')
    axs[0].set_xlabel('harmonic order $h$')
    axs[0].set_ylabel(f'magnitude {unit}')
    axs[0].set_title('Magnitude spectrum' if title is None else f'{title} -- magnitude')
    axs[0].set_xlim([0, h_max])

    axs[1].stem(h[mask], phase, linefmt='tab:red', markerfmt='o', basefmt=' ')
    axs[1].set_xlabel('harmonic order $h$')
    axs[1].set_ylabel('phase (deg)')
    axs[1].set_title('Phase spectrum' if title is None else f'{title} -- phase')
    axs[1].set_xlim([0, h_max])

    plt.tight_layout()
    plt.show()

### DFT Evaluation Window

The DFT is evaluated with $N=2048$ uniformly spaced samples taken over the final fundamental cycle, which is treated as steady state.

In [ ]:
N_dft = 2048
t_window = np.linspace(t_end - 1/f1, t_end, N_dft, endpoint=False)

vp_window = Vdc/2 * q_signal(t_window)
i_f_window = sol.sol(t_window)[0]

vp_Ck_p = dft_peak(vp_window)
print("Pole voltage v_ao:")
compute_thd(vp_Ck_p)
plot_spectrum(vp_Ck_p, h_max=200, title='Pole voltage $v_{ao}$', unit='(V)')

i_f_Ck_p = dft_peak(i_f_window)
print()
print("Injected current i_f:")
compute_thd(i_f_Ck_p)
plot_spectrum(i_f_Ck_p, h_max=200, title='Injected current $i_f$', unit='(A)')

### Credits and Disclaimer

This notebook is part of the NPTEL course *Grid Connected Power Converters - Operating Principles*.

The simulation ideas, modelling approach, and technical interpretation in this notebook are original to the author.

This notebook presents an idealized switching-pole converter model. It assumes a balanced DC link and other simplifying idealizations that are appropriate for pedagogical study of waveforms and harmonic content. The results are therefore useful for understanding the operating principle and analysis method, but they are not a substitute for detailed converter design validation.

AI assistance was used only to polish and refine portions of the code structure and documentation language.